# Notebook 04a: Bayesian Optimization (TPE)

This notebook extends Notebook 03 by keeping the baseline preprocessing contract fixed (selected features, optimal scaler per model, and Stratified 5-fold CV) and tuning model hyperparameters with Optuna TPE on training data only.

Generated outputs:
- `TPE_trials_full.csv`
- `TPE_convergence.csv`
- `TPE_best_params.csv`
- `TPE_optimal_fold_scores.csv`
- `TPE_optimization_time.csv`
- `TPE_test_results.csv` (optional section enabled below)

In [5]:
from __future__ import annotations

import json
import random
import time
from pathlib import Path
from typing import Any, Dict, List

import numpy as np
import pandas as pd

import optuna
from optuna.samplers import TPESampler

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, PowerTransformer, QuantileTransformer
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    log_loss,
    cohen_kappa_score,
    matthews_corrcoef,
)

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier, GradientBoostingClassifier

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

RANDOM_SEED = 42
N_TRIALS_PER_MODEL = 50
INCLUDE_FINAL_TEST_EVAL = True

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

NOTEBOOK_DIR = Path.cwd()
RESULTS_DIR = (NOTEBOOK_DIR / '../results/tables').resolve()

SEARCH_SPACE_PATH = RESULTS_DIR / 'unified_search_space_clean.csv'
if not SEARCH_SPACE_PATH.exists():
    SEARCH_SPACE_PATH = RESULTS_DIR / 'unified_search_space.csv'

TRIALS_FULL_PATH = RESULTS_DIR / 'TPE_trials_full.csv'
CONVERGENCE_PATH = RESULTS_DIR / 'TPE_convergence.csv'
BEST_PARAMS_PATH = RESULTS_DIR / 'TPE_best_params.csv'
FOLD_SCORES_PATH = RESULTS_DIR / 'TPE_optimal_fold_scores.csv'
OPT_TIME_PATH = RESULTS_DIR / 'TPE_optimization_time.csv'
TEST_RESULTS_PATH = RESULTS_DIR / 'TPE_test_results.csv'
OPTUNA_DB_PATH = RESULTS_DIR / 'tpe_optuna_studies.db'

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

print(f'RANDOM_SEED = {RANDOM_SEED}')
print('CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)')
print(f'Results directory: {RESULTS_DIR}')
print(f'Search space file: {SEARCH_SPACE_PATH.name}')

RANDOM_SEED = 42
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
Results directory: C:\Users\omar8\Desktop\2026\Master & Research\smart-grid-stability-v01\results\tables
Search space file: unified_search_space_clean.csv


In [2]:
required_files = [
    RESULTS_DIR / 'X_train.npy',
    RESULTS_DIR / 'X_test.npy',
    RESULTS_DIR / 'y_train.npy',
    RESULTS_DIR / 'y_test.npy',
    RESULTS_DIR / 'selected_features.json',
    RESULTS_DIR / 'feature_names.csv',
    RESULTS_DIR / 'train_test_split_info.json',
    RESULTS_DIR / 'baseline_optimal_scalers.csv',
    SEARCH_SPACE_PATH,
]

missing = [str(p) for p in required_files if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required inputs:\n' + '\n'.join(missing))

X_train = np.load(RESULTS_DIR / 'X_train.npy')
X_test = np.load(RESULTS_DIR / 'X_test.npy')
y_train = np.load(RESULTS_DIR / 'y_train.npy')
y_test = np.load(RESULTS_DIR / 'y_test.npy')

with open(RESULTS_DIR / 'selected_features.json', 'r', encoding='utf-8') as f:
    selected_features = json.load(f)

with open(RESULTS_DIR / 'train_test_split_info.json', 'r', encoding='utf-8') as f:
    split_info = json.load(f)

feature_names_df = pd.read_csv(RESULTS_DIR / 'feature_names.csv')
if 'feature' not in feature_names_df.columns:
    raise ValueError('feature_names.csv must contain a feature column')

all_feature_names = feature_names_df['feature'].tolist()
feature_to_idx = {name: idx for idx, name in enumerate(all_feature_names)}
selected_idx = [feature_to_idx[f] for f in selected_features if f in feature_to_idx]

if len(selected_idx) != len(selected_features):
    missing_features = sorted(set(selected_features) - set(feature_to_idx.keys()))
    raise ValueError(f'Selected features not found in feature_names.csv: {missing_features}')

X_train_sel = X_train[:, selected_idx]
X_test_sel = X_test[:, selected_idx]

baseline_scalers_df = pd.read_csv(RESULTS_DIR / 'baseline_optimal_scalers.csv')
required_scaler_cols = {'Model', 'Optimal_Scaler'}
if not required_scaler_cols.issubset(baseline_scalers_df.columns):
    raise ValueError(f'baseline_optimal_scalers.csv missing columns: {required_scaler_cols}')

search_df = pd.read_csv(SEARCH_SPACE_PATH)
required_search_cols = {'Model', 'Hyperparameter', 'Type', 'Lower_Bound', 'Upper_Bound', 'Choices', 'Log_Scale'}
if not required_search_cols.issubset(search_df.columns):
    raise ValueError(f'{SEARCH_SPACE_PATH.name} missing columns: {required_search_cols}')

print(f'X_train_sel shape: {X_train_sel.shape}')
print(f'X_test_sel shape: {X_test_sel.shape}')
print(f'Selected features: {len(selected_features)}')
print(f'Train/Test split info random_state: {split_info.get("random_state", "NA")}')

X_train_sel shape: (48000, 13)
X_test_sel shape: (12000, 13)
Selected features: 13
Train/Test split info random_state: 42


In [3]:
scalers = {
    'Raw': None,
    'Standard': StandardScaler(),
    'MinMax': MinMaxScaler(),
    'Robust': RobustScaler(),
    'Power': PowerTransformer(),
    'Quantile': QuantileTransformer(output_distribution='normal', random_state=RANDOM_SEED),
}

model_alias_to_code = {
    'LogisticRegression': 'LR',
    'LDA': 'LDA',
    'QDA': 'QDA',
    'NaiveBayes': 'NB',
    'KNN': 'KNN',
    'LinearSVC': 'LinearSVC',
    'SVC': 'SVM',
    'AdaBoost': 'AdaBoost',
    'RandomForest': 'RF',
    'GradientBoosting': 'GB',
    'XGBoost': 'XGB',
    'LightGBM': 'LGBM',
    'CatBoost': 'CB',
    'SGD': 'SGD',
}

code_to_model_alias = {v: k for k, v in model_alias_to_code.items()}

def build_model(model_code: str, params: Dict[str, Any]) -> Any:
    p = dict(params)

    if model_code == 'LR':
        penalty = p.get('penalty', 'l2')
        solver = 'liblinear' if penalty == 'l1' else 'lbfgs'
        return LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, solver=solver, **p)

    if model_code == 'LDA':
        return LinearDiscriminantAnalysis(**p)

    if model_code == 'QDA':
        return QuadraticDiscriminantAnalysis(**p)

    if model_code == 'NB':
        return GaussianNB(**p)

    if model_code == 'KNN':
        return KNeighborsClassifier(**p)

    if model_code == 'LinearSVC':
        if p.get('loss') == 'hinge':
            p['dual'] = True
        else:
            p['dual'] = 'auto'
        return LinearSVC(random_state=RANDOM_SEED, **p)

    if model_code == 'SVM':
        return SVC(probability=True, random_state=RANDOM_SEED, **p)

    if model_code == 'AdaBoost':
        if p.get('algorithm') == 'SAMME.R':
            p['algorithm'] = 'SAMME'
        return AdaBoostClassifier(random_state=RANDOM_SEED, **p)

    if model_code == 'RF':
        if p.get('max_features') == 'None':
            p['max_features'] = None
        return RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1, **p)

    if model_code == 'GB':
        return GradientBoostingClassifier(random_state=RANDOM_SEED, **p)

    if model_code == 'XGB':
        return xgb.XGBClassifier(eval_metric='logloss', random_state=RANDOM_SEED, verbosity=0, n_jobs=-1, **p)

    if model_code == 'LGBM':
        return lgb.LGBMClassifier(random_state=RANDOM_SEED, verbose=-1, n_jobs=-1, **p)

    if model_code == 'CB':
        return CatBoostClassifier(verbose=0, random_state=RANDOM_SEED, **p)

    if model_code == 'SGD':
        return SGDClassifier(max_iter=1000, random_state=RANDOM_SEED, **p)

    raise ValueError(f'Unsupported model code: {model_code}')

def parse_model_search_space(search_space_df: pd.DataFrame) -> Dict[str, List[Dict[str, Any]]]:
    grouped: Dict[str, List[Dict[str, Any]]] = {}
    for alias_name, g in search_space_df.groupby('Model'):
        model_code = model_alias_to_code.get(alias_name)
        if model_code is None:
            continue

        entries: List[Dict[str, Any]] = []
        for _, row in g.iterrows():
            hp_type = str(row['Type']).strip().lower()
            log_scale_raw = row.get('Log_Scale', False)
            if isinstance(log_scale_raw, str):
                log_scale_flag = log_scale_raw.strip().lower() == 'true'
            else:
                log_scale_flag = bool(log_scale_raw)

            entry = {
                'name': str(row['Hyperparameter']).strip(),
                'type': hp_type,
                'low': row.get('Lower_Bound', np.nan),
                'high': row.get('Upper_Bound', np.nan),
                'choices': row.get('Choices', np.nan),
                'log_scale': log_scale_flag,
            }
            entries.append(entry)
        grouped[model_code] = entries
    return grouped

model_spaces = parse_model_search_space(search_df)
optimal_scaler_by_model = dict(zip(baseline_scalers_df['Model'], baseline_scalers_df['Optimal_Scaler']))

target_models = [m for m in baseline_scalers_df['Model'].tolist() if m in model_spaces]
missing_spaces = [m for m in baseline_scalers_df['Model'].tolist() if m not in model_spaces]
if missing_spaces:
    print('Models with no search-space rows (will be skipped):', missing_spaces)

def suggest_params(trial: optuna.trial.Trial, model_code: str) -> Dict[str, Any]:
    params: Dict[str, Any] = {}
    for entry in model_spaces.get(model_code, []):
        name = entry['name']
        typ = entry['type']

        if typ == 'categorical':
            choices_raw = str(entry['choices'])
            choices = [c.strip() for c in choices_raw.split('|') if c.strip()]
            params[name] = trial.suggest_categorical(name, choices)
        elif typ == 'int':
            low = int(float(entry['low']))
            high = int(float(entry['high']))
            params[name] = trial.suggest_int(name, low, high)
        elif typ == 'float':
            low = float(entry['low'])
            high = float(entry['high'])
            params[name] = trial.suggest_float(name, low, high, log=bool(entry['log_scale']))
        else:
            raise ValueError(f'Unsupported hyperparameter type: {typ} for {model_code}:{name}')

    return params

def make_pipeline(model_code: str, model_params: Dict[str, Any], scaler_name: str) -> Pipeline:
    model = build_model(model_code=model_code, params=model_params)
    scaler_obj = scalers[scaler_name]

    if scaler_obj is None:
        return Pipeline([('model', model)])

    return Pipeline([('scaler', clone(scaler_obj)), ('model', model)])

print(f'Models to optimize: {target_models}')
print(f'Number of trials/model: {N_TRIALS_PER_MODEL}')

Models to optimize: ['CB', 'XGB', 'SVM', 'LGBM', 'RF', 'KNN', 'GB', 'LinearSVC', 'LDA', 'LR', 'QDA', 'SGD', 'AdaBoost', 'NB']
Number of trials/model: 50


In [4]:
def load_or_init_trials_df(path: Path) -> pd.DataFrame:
    cols = ['Model', 'Trial', 'CV_Mean', 'Best_Score_So_Far', 'Trial_Runtime_Sec']
    if path.exists():
        df = pd.read_csv(path)
        for c in cols:
            if c not in df.columns:
                raise ValueError(f'{path.name} missing required column: {c}')
        return df[cols].copy()
    return pd.DataFrame(columns=cols)

def save_trials_and_convergence(trials_df: pd.DataFrame) -> None:
    if trials_df.empty:
        trials_df.to_csv(TRIALS_FULL_PATH, index=False)
        pd.DataFrame(columns=['Model', 'Trial', 'Best_Score_So_Far']).to_csv(CONVERGENCE_PATH, index=False)
        return

    clean = trials_df.drop_duplicates(subset=['Model', 'Trial'], keep='last').sort_values(['Model', 'Trial'])
    clean.to_csv(TRIALS_FULL_PATH, index=False)

    conv = clean[['Model', 'Trial', 'Best_Score_So_Far']].copy()
    conv.to_csv(CONVERGENCE_PATH, index=False)

def load_or_init_best_params(path: Path) -> pd.DataFrame:
    fixed_cols = ['Model', 'Optimal_Scaler', 'Best_CV_Mean', 'Best_Trial']
    if path.exists():
        df = pd.read_csv(path)
        for c in fixed_cols:
            if c not in df.columns:
                raise ValueError(f'{path.name} missing required column: {c}')
        return df
    return pd.DataFrame(columns=fixed_cols)

def upsert_best_row(best_df: pd.DataFrame, row: Dict[str, Any]) -> pd.DataFrame:
    row_df = pd.DataFrame([row])
    if best_df.empty:
        return row_df

    all_cols = sorted(set(best_df.columns) | set(row_df.columns))
    best_df = best_df.reindex(columns=all_cols)
    row_df = row_df.reindex(columns=all_cols)

    best_df = best_df[best_df['Model'] != row['Model']]
    out = pd.concat([best_df, row_df], ignore_index=True)
    out = out.sort_values('Model').reset_index(drop=True)
    return out

trials_df = load_or_init_trials_df(TRIALS_FULL_PATH)
best_params_df = load_or_init_best_params(BEST_PARAMS_PATH)

storage_uri = f'sqlite:///{OPTUNA_DB_PATH.as_posix()}'

for model_code in target_models:
    scaler_name = optimal_scaler_by_model[model_code]

    model_trials = trials_df[trials_df['Model'] == model_code].sort_values('Trial')
    completed = int(model_trials['Trial'].nunique())

    if completed >= N_TRIALS_PER_MODEL and (best_params_df['Model'] == model_code).any():
        print(f'Skip {model_code}: already complete ({completed}/{N_TRIALS_PER_MODEL})')
        continue

    print(f'Optimizing {model_code} with scaler={scaler_name} | completed={completed}/{N_TRIALS_PER_MODEL}')

    sampler = TPESampler(seed=RANDOM_SEED)
    study = optuna.create_study(
        direction='maximize',
        sampler=sampler,
        study_name=f'TPE_{model_code}',
        storage=storage_uri,
        load_if_exists=True,
    )

    best_so_far = float(model_trials['Best_Score_So_Far'].max()) if not model_trials.empty else float('-inf')
    next_trial_idx = int(model_trials['Trial'].max()) + 1 if not model_trials.empty else 1

    best_so_far_holder = [best_so_far]
    next_trial_idx_holder = [next_trial_idx]

    def objective(trial: optuna.trial.Trial) -> float:
        t0 = time.perf_counter()
        params = suggest_params(trial, model_code=model_code)
        pipeline = make_pipeline(model_code=model_code, model_params=params, scaler_name=scaler_name)

        cv_out = cross_validate(
            pipeline,
            X_train_sel,
            y_train,
            cv=cv,
            scoring='accuracy',
            n_jobs=-1,
            return_train_score=False,
        )

        mean_score = float(np.mean(cv_out['test_score']))
        trial.set_user_attr('runtime_sec', float(time.perf_counter() - t0))
        return mean_score

    def on_trial_complete(*args: Any) -> None:
        trial_obj = args[1]
        if trial_obj.state != optuna.trial.TrialState.COMPLETE:
            return

        score = float(trial_obj.value)
        runtime_sec = float(trial_obj.user_attrs.get('runtime_sec', np.nan))
        best_so_far_holder[0] = max(best_so_far_holder[0], score)

        new_row = {
            'Model': model_code,
            'Trial': next_trial_idx_holder[0],
            'CV_Mean': score,
            'Best_Score_So_Far': best_so_far_holder[0],
            'Trial_Runtime_Sec': runtime_sec,
        }
        next_trial_idx_holder[0] += 1

        trials_df.loc[len(trials_df)] = new_row
        save_trials_and_convergence(trials_df)

    remaining = N_TRIALS_PER_MODEL - completed
    if remaining > 0:
        study.optimize(objective, n_trials=remaining, callbacks=[on_trial_complete], n_jobs=1, show_progress_bar=False)

    best_trial = study.best_trial
    best_row: Dict[str, Any] = {
        'Model': model_code,
        'Optimal_Scaler': scaler_name,
        'Best_CV_Mean': float(best_trial.value),
        'Best_Trial': int(best_trial.number),
    }
    best_row.update(best_trial.params)

    best_params_df = upsert_best_row(best_params_df, best_row)
    best_params_df.to_csv(BEST_PARAMS_PATH, index=False)

    print(f'  done {model_code} | best={best_trial.value:.6f}')

save_trials_and_convergence(trials_df)
best_params_df.to_csv(BEST_PARAMS_PATH, index=False)

print('Saved:')
print(f' - {TRIALS_FULL_PATH.name}')
print(f' - {CONVERGENCE_PATH.name}')
print(f' - {BEST_PARAMS_PATH.name}')

Optimizing CB with scaler=Raw | completed=0/50


[I 2026-03-18 21:23:01,567] A new study created in RDB with name: TPE_CB
[I 2026-03-18 21:23:25,421] Trial 0 finished with value: 0.9837291666666665 and parameters: {'bagging_temperature': 3.745401188473625, 'border_count': 244, 'depth': 8, 'iterations': 319, 'l2_leaf_reg': 2.5361081166471375e-07, 'learning_rate': 0.08643731496473929}. Best is trial 0 with value: 0.9837291666666665.
[I 2026-03-18 21:23:45,043] Trial 1 finished with value: 0.9879583333333333 and parameters: {'bagging_temperature': 0.5808361216819946, 'border_count': 226, 'depth': 7, 'iterations': 369, 'l2_leaf_reg': 1.5320059381854043e-08, 'learning_rate': 0.48525582755937724}. Best is trial 1 with value: 0.9879583333333333.
[I 2026-03-18 21:23:47,910] Trial 2 finished with value: 0.9624375000000001 and parameters: {'bagging_temperature': 8.324426408004218, 'border_count': 79, 'depth': 4, 'iterations': 132, 'l2_leaf_reg': 5.472429642032198e-06, 'learning_rate': 0.26713065149979653}. Best is trial 1 with value: 0.9879583

  done CB | best=0.995125
Optimizing XGB with scaler=Power | completed=0/50


[I 2026-03-18 21:52:15,640] A new study created in RDB with name: TPE_XGB
[I 2026-03-18 21:52:21,891] Trial 0 finished with value: 0.9713749999999999 and parameters: {'colsample_bytree': 0.6872700594236812, 'gamma': 4.75357153204958, 'learning_rate': 0.3686770314875885, 'max_depth': 10, 'min_child_weight': 2, 'n_estimators': 120, 'reg_alpha': 3.3323645788192616e-08, 'reg_lambda': 0.6245760287469893, 'subsample': 0.8005575058716043}. Best is trial 0 with value: 0.9713749999999999.
[I 2026-03-18 21:52:31,825] Trial 1 finished with value: 0.9851458333333334 and parameters: {'colsample_bytree': 0.8540362888980227, 'gamma': 0.10292247147901223, 'learning_rate': 0.48525582755937724, 'max_depth': 13, 'min_child_weight': 3, 'n_estimators': 132, 'reg_alpha': 4.4734294104626844e-07, 'reg_lambda': 5.472429642032198e-06, 'subsample': 0.762378215816119}. Best is trial 1 with value: 0.9851458333333334.
[I 2026-03-18 21:52:39,677] Trial 2 finished with value: 0.9845 and parameters: {'colsample_bytree

  done XGB | best=0.990958
Optimizing SVM with scaler=Standard | completed=0/50


...

  done SGD | best=0.896229
Optimizing AdaBoost with scaler=MinMax | completed=0/50


[I 2026-03-19 08:13:11,714] A new study created in RDB with name: TPE_AdaBoost
[I 2026-03-19 08:14:10,138] Trial 0 finished with value: 0.9129166666666666 and parameters: {'algorithm': 'SAMME.R', 'learning_rate': 0.48343714531846416, 'n_estimators': 319}. Best is trial 0 with value: 0.9129166666666666.
[I 2026-03-19 08:15:27,521] Trial 1 finished with value: 0.8474166666666667 and parameters: {'algorithm': 'SAMME', 'learning_rate': 0.013603546136118085, 'n_estimators': 440}. Best is trial 0 with value: 0.9129166666666666.
[I 2026-03-19 08:16:53,608] Trial 2 finished with value: 0.8461666666666667 and parameters: {'algorithm': 'SAMME.R', 'learning_rate': 0.011152328125494349, 'n_estimators': 487}. Best is trial 0 with value: 0.9129166666666666.
[I 2026-03-19 08:17:21,052] Trial 3 finished with value: 0.8406041666666667 and parameters: {'algorithm': 'SAMME', 'learning_rate': 0.026205032550962556, 'n_estimators': 132}. Best is trial 0 with value: 0.9129166666666666.
[I 2026-03-19 08:17:53

  done AdaBoost | best=0.950292
Optimizing NB with scaler=Power | completed=0/50


[I 2026-03-19 09:08:02,248] Trial 0 finished with value: 0.8430416666666666 and parameters: {'var_smoothing': 1.7670169402947963e-10}. Best is trial 0 with value: 0.8430416666666666.
[I 2026-03-19 09:08:03,877] Trial 1 finished with value: 0.8430416666666666 and parameters: {'var_smoothing': 5.061576888752309e-07}. Best is trial 0 with value: 0.8430416666666666.
[I 2026-03-19 09:08:06,259] Trial 2 finished with value: 0.8430416666666666 and parameters: {'var_smoothing': 2.4658329458549168e-08}. Best is trial 0 with value: 0.8430416666666666.
[I 2026-03-19 09:08:08,027] Trial 3 finished with value: 0.8430416666666666 and parameters: {'var_smoothing': 3.9079671568228905e-09}. Best is trial 0 with value: 0.8430416666666666.
[I 2026-03-19 09:08:10,044] Trial 4 finished with value: 0.8430416666666666 and parameters: {'var_smoothing': 8.632008168602543e-12}. Best is trial 0 with value: 0.8430416666666666.
[I 2026-03-19 09:08:11,957] Trial 5 finished with value: 0.8430416666666666 and paramet

  done NB | best=0.843042
Saved:
 - TPE_trials_full.csv
 - TPE_convergence.csv
 - TPE_best_params.csv


In [6]:
best_params_df = pd.read_csv(BEST_PARAMS_PATH)

base_cols = {'Model', 'Optimal_Scaler', 'Best_CV_Mean', 'Best_Trial'}
float_keys = {
    'learning_rate', 'subsample', 'colsample_bytree', 'reg_alpha', 'reg_lambda',
    'l2_leaf_reg', 'alpha', 'l1_ratio', 'reg_param', 'var_smoothing', 'C', 'coef0'
}

fold_rows: List[Dict[str, Any]] = []
for _, row in best_params_df.iterrows():
    model_code = row['Model']
    scaler_name = row['Optimal_Scaler']

    params: Dict[str, Any] = {}
    for col in best_params_df.columns:
        if col in base_cols:
            continue
        value = row[col]
        if pd.notna(value):
            params[col] = value

    for k, v in list(params.items()):
        if isinstance(v, float) and v.is_integer() and k not in float_keys:
            params[k] = int(v)

    pipeline = make_pipeline(model_code=model_code, model_params=params, scaler_name=scaler_name)
    cv_out = cross_validate(
        pipeline,
        X_train_sel,
        y_train,
        cv=cv,
        scoring='accuracy',
        n_jobs=-1,
        return_train_score=False,
    )

    fold_scores = cv_out['test_score']
    fold_row: Dict[str, Any] = {
        'Model': model_code,
        'Optimal_Scaler': scaler_name,
    }
    for i, s in enumerate(fold_scores, start=1):
        fold_row[f'Fold_{i}'] = float(s)

    fold_row['Mean'] = float(np.mean(fold_scores))
    fold_row['Std'] = float(np.std(fold_scores, ddof=0))
    fold_rows.append(fold_row)

fold_scores_df = pd.DataFrame(fold_rows).sort_values('Model').reset_index(drop=True)
fold_scores_df.to_csv(FOLD_SCORES_PATH, index=False)

trials_df = pd.read_csv(TRIALS_FULL_PATH)
opt_time_df = (
    trials_df.groupby('Model', as_index=False)
    .agg(
        Number_of_Trials=('Trial', 'count'),
        Total_Optimization_Time_Sec=('Trial_Runtime_Sec', 'sum'),
        Mean_Trial_Time_Sec=('Trial_Runtime_Sec', 'mean'),
    )
    .sort_values('Model')
    .reset_index(drop=True)
)
opt_time_df.to_csv(OPT_TIME_PATH, index=False)

print(f'Saved: {FOLD_SCORES_PATH.name} ({len(fold_scores_df)} rows)')
print(f'Saved: {OPT_TIME_PATH.name} ({len(opt_time_df)} rows)')

Saved: TPE_optimal_fold_scores.csv (14 rows)
Saved: TPE_optimization_time.csv (14 rows)


In [7]:
if INCLUDE_FINAL_TEST_EVAL:
    best_params_df = pd.read_csv(BEST_PARAMS_PATH)
    base_cols = {'Model', 'Optimal_Scaler', 'Best_CV_Mean', 'Best_Trial'}

    test_rows: List[Dict[str, Any]] = []
    for _, row in best_params_df.iterrows():
        model_code = row['Model']
        scaler_name = row['Optimal_Scaler']

        params: Dict[str, Any] = {}
        for col in best_params_df.columns:
            if col in base_cols:
                continue
            value = row[col]
            if pd.notna(value):
                params[col] = value

        for k, v in list(params.items()):
            if isinstance(v, float) and v.is_integer() and k not in float_keys:
                params[k] = int(v)

        model = build_model(model_code=model_code, params=params)
        scaler_obj = scalers[scaler_name]

        if scaler_obj is None:
            Xtr = X_train_sel
            Xte = X_test_sel
        else:
            s = clone(scaler_obj)
            Xtr = s.fit_transform(X_train_sel)
            Xte = s.transform(X_test_sel)

        model.fit(Xtr, y_train)
        y_pred = model.predict(Xte)

        acc = accuracy_score(y_test, y_pred)
        prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary', zero_division=0)
        kappa = cohen_kappa_score(y_test, y_pred)
        mcc = matthews_corrcoef(y_test, y_pred)

        if hasattr(model, 'predict_proba'):
            y_score = model.predict_proba(Xte)[:, 1]
            auc = roc_auc_score(y_test, y_score)
            ll = log_loss(y_test, y_score)
        elif hasattr(model, 'decision_function'):
            y_score = model.decision_function(Xte)
            auc = roc_auc_score(y_test, y_score)
            ll = np.nan
        else:
            auc = np.nan
            ll = np.nan

        test_rows.append({
            'Model': model_code,
            'Optimal_Scaler': scaler_name,
            'Accuracy': float(acc),
            'Precision': float(prec),
            'Recall': float(rec),
            'F1': float(f1),
            'AUC': float(auc) if pd.notna(auc) else np.nan,
            'LogLoss': float(ll) if pd.notna(ll) else np.nan,
            'Kappa': float(kappa),
            'MCC': float(mcc),
        })

    tpe_test_df = pd.DataFrame(test_rows).sort_values('Accuracy', ascending=False).reset_index(drop=True)
    tpe_test_df.to_csv(TEST_RESULTS_PATH, index=False)
    print(f'Saved: {TEST_RESULTS_PATH.name} ({len(tpe_test_df)} rows)')
    print('Metrics columns: Accuracy, Precision, Recall, F1, AUC, LogLoss, Kappa, MCC')
else:
    print('Final test evaluation skipped by configuration.')

c:\Users\omar8\miniconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
c:\Users\omar8\miniconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\omar8\miniconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Saved: TPE_test_results.csv (14 rows)
Metrics columns: Accuracy, Precision, Recall, F1, AUC, LogLoss, Kappa, MCC


In [8]:
expected_outputs = [
    TRIALS_FULL_PATH,
    CONVERGENCE_PATH,
    BEST_PARAMS_PATH,
    FOLD_SCORES_PATH,
    OPT_TIME_PATH,
]
if INCLUDE_FINAL_TEST_EVAL:
    expected_outputs.append(TEST_RESULTS_PATH)

print('Generated outputs:')
for p in expected_outputs:
    status = 'OK' if p.exists() else 'MISSING'
    rows = len(pd.read_csv(p)) if p.exists() else 0
    print(f' - {p.name}: {status}, rows={rows}')

Generated outputs:
 - TPE_trials_full.csv: OK, rows=700
 - TPE_convergence.csv: OK, rows=700
 - TPE_best_params.csv: OK, rows=14
 - TPE_optimal_fold_scores.csv: OK, rows=14
 - TPE_optimization_time.csv: OK, rows=14
 - TPE_test_results.csv: OK, rows=14
